In [1]:
import pandas as pd

student = pd.read_csv("student-mat.csv", sep=";")


In [2]:
student.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [4]:
print("Dataset shape:", student.shape)
print("\nColumns:")
print(student.columns.tolist())


Dataset shape: (395, 33)

Columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [5]:
print("\nMissing values:")
print(student.isnull().sum())


Missing values:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G1            0
G2            0
G3            0
dtype: int64


In [8]:
student["pass"] = (student["G3"] >= 10).astype(int)

print(student[["G3", "pass"]].head(10))

print("\nClass counts:")
print(student["pass"].value_counts())

print("\nClass percentages:")
print(student["pass"].value_counts(normalize=True) * 100)

   G3  pass
0   6     0
1   6     0
2  10     1
3  15     1
4  10     1
5  15     1
6  11     1
7   6     0
8  19     1
9  15     1

Class counts:
pass
1    265
0    130
Name: count, dtype: int64

Class percentages:
pass
1    67.088608
0    32.911392
Name: proportion, dtype: float64


In [9]:
X = student.drop(columns=["G1", "G2", "G3", "pass"])
y = student["pass"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

Predictor shape: (395, 30)
Target shape: (395,)


In [11]:
X_encoded = pd.get_dummies(X, drop_first=True)

print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)

X_encoded.head()

Before encoding: (395, 30)
After encoding: (395, 39)


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,True,False,True,False,False,False,True,True,False,False
1,17,1,1,1,2,0,5,3,3,1,...,False,False,False,True,False,False,False,True,True,False
2,15,1,1,1,2,3,4,3,2,2,...,True,False,True,False,True,False,True,True,True,False
3,15,4,2,1,3,0,3,2,2,1,...,True,False,False,True,True,True,True,True,True,True
4,16,3,3,1,2,0,4,3,2,1,...,False,False,False,True,True,False,True,True,False,False


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (316, 39)
Testing size: (79, 39)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [16]:
y_pred_log = log_model.predict(X_test_scaled)

print(y_pred_log)

[0 0 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 1 1 1 1 0 1 1 0 1 1 1 1 0 1 1 1 1 1 1 0
 0 1 0 0 0 0 1 1 1 0 1 0 1 0 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1
 1 1 1 1 1]


In [17]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

accuracy = accuracy_score(y_test, y_pred_log)
f1 = f1_score(y_test, y_pred_log, average="macro")

print("Accuracy:", round(accuracy, 3))
print("Macro F1:", round(f1, 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

Accuracy: 0.633
Macro F1: 0.561

Confusion Matrix:
[[ 9 17]
 [12 41]]

Classification Report:
              precision    recall  f1-score   support

           0       0.43      0.35      0.38        26
           1       0.71      0.77      0.74        53

    accuracy                           0.63        79
   macro avg       0.57      0.56      0.56        79
weighted avg       0.62      0.63      0.62        79



In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
y_pred_rf = rf_model.predict(X_test)

In [20]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average="macro")

print("Random Forest")
print("Accuracy:", round(rf_accuracy, 3))
print("Macro F1:", round(rf_f1, 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest
Accuracy: 0.684
Macro F1: 0.559

Confusion Matrix:
[[ 6 20]
 [ 5 48]]

Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.23      0.32        26
           1       0.71      0.91      0.79        53

    accuracy                           0.68        79
   macro avg       0.63      0.57      0.56        79
weighted avg       0.65      0.68      0.64        79



In [21]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(10))

     Feature  Importance
5   failures    0.094421
12  absences    0.093110
8      goout    0.056362
0        age    0.049542
11    health    0.046366
7   freetime    0.043624
10      Walc    0.042256
2       Fedu    0.041583
1       Medu    0.037975
6     famrel    0.033760
